## Stacks — the Compose file in cluster form

Typing 15-flag `docker service create` lines is the Swarm version of typing `docker run` instead of using Compose. The fix is the same too: **a declarative file.** A **stack** is a Compose-format YAML with an extra **`deploy:`** block, deployed in one command:

```bash
docker stack deploy -c stack.yaml my-app
```

Everything you know from Compose (module 06) carries over — `image`, `ports`, `networks`, `volumes`, `secrets` — and the new `deploy:` key holds the Swarm-specific policy:

```yaml
services:
  web:
    image: myorg/web:1.2.3
    ports: ["8080:80"]
    networks: [app-net]
    deploy:
      replicas: 3
      update_config: { parallelism: 1, delay: 10s, failure_action: rollback, monitor: 30s }
      restart_policy: { condition: on-failure }
      resources: { limits: { cpus: "0.5", memory: 256M } }
      placement:
        constraints: [ "node.role == worker" ]

networks:
  app-net: { driver: overlay }
```

The mental mapping is clean: **a service's *what* (image, ports, networks) stays where it was in Compose; its *how-in-a-cluster* (replicas, rolling-update policy, resource limits, placement) moves into `deploy:`.** That `deploy:` block is *ignored* by plain `docker compose up` and *only* honored by `docker stack deploy`, which is exactly why the same file can often serve both local dev and cluster deploy. Stacks are how real Swarm is operated — you version the YAML in git, `stack deploy` to apply, and `docker stack ls` / `ps` / `rm` to manage the whole app as one unit rather than service by service.